# ⚡ Module 09: FastAPI in Production (def vs async def, Lifespans)

FastAPI is the world's most popular async Python framework, but it has one famous architectural nuance:

### 🚨 The `def` vs `async def` FastAPI Trap:
- If you write `def endpoint()`: FastAPI **runs it in a separate threadpool** (safe for blocking code!).
- If you write `async def endpoint()`: FastAPI **runs it directly on the main event loop**!
- **The Disaster**: If you run blocking code (like `time.sleep` or synchronous DB calls) inside `async def`, you freeze the server for every single concurrent user!

### What We Will Master:
1. **`def` vs `async def` threadpool mechanics**.
2. **Async Dependency Injection** (`Depends`) with automatic session cleanup.
3. **Modern Lifespan Managers** (`@asynccontextmanager` replacing deprecated startup events).
4. **FastAPI BackgroundTasks**.
5. **Streaming Responses (SSE)** for Real-Time LLM Token Streaming.

In [1]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app_state: dict):
    # 1. Startup phase
    print("🚀 Simulating FastAPI App Lifespan:")
    print("  [Startup] Initializing Async Database Connection Pool...")
    await asyncio.sleep(0.1)
    print("  [Startup] Pre-warming Redis and ML Vector Cache...")
    app_state["db_pool"] = "Ready"
    yield # App is running and serving requests
    # 2. Shutdown phase
    print("  [Shutdown] Draining background queues and closing DB connections safely.")
    await asyncio.sleep(0.05)

state = {}
async with lifespan(state):
    print("  [Runtime] FastAPI app is live and handling requests!")
print("✅ Lifespan cycle verified!")

🚀 Simulating FastAPI App Lifespan:
  [Startup] Initializing Async Database Connection Pool...
  [Startup] Pre-warming Redis and ML Vector Cache...
  [Runtime] FastAPI app is live and handling requests!
  [Shutdown] Draining background queues and closing DB connections safely.
✅ Lifespan cycle verified!


---
## 2. Streaming LLM Tokens in FastAPI with Async Generators (SSE)

How ChatGPT and modern AI applications stream text in real-time using FastAPI's `StreamingResponse`.

In [2]:
async def fake_llm_stream():
    tokens = ["Async", " Python", " is", " lightning", " fast!"]
    for t in tokens:
        await asyncio.sleep(0.1) # Simulates LLM inference latency per token
        yield f"data: {{'token': '{t}'}}\n\n"
    yield "data: [DONE]\n\n"

print("Client connected to SSE stream: http://api/v1/chat/completions/stream")
async for chunk in fake_llm_stream():
    print(chunk.strip())

Client connected to SSE stream: http://api/v1/chat/completions/stream
data: {'token': 'Async'}
data: {'token': ' Python'}
data: {'token': ' is'}
data: {'token': ' lightning'}
data: {'token': ' fast!'}
data: [DONE]
✅ Stream completed without waiting for full response generation!
